# Backtest

Loads all `WalletSelectionStrategy` objects from the workspace registry,
scores test signals for each strategy's wallet set, applies each strategy's
trigger function, and runs a full strategy sweep.  Results are compared in a
summary table and cumulative PnL plot.

**No strategy logic is defined here** — all trigger rules and parameters
live in the persisted strategy registry (written by `stage1_wallet_strategy_selection`).


In [1]:
%load_ext autoreload
%autoreload 2

import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.dataset as ds


## Configuration

In [2]:
PRICE_BUCKET_BINS   = [0.0, 0.1, 0.25, 0.4, 0.6, 0.75, 0.9, 1.0]
PRICE_BUCKET_LABELS = [
    "0.00-0.10", "0.10-0.25", "0.25-0.40", "0.40-0.60",
    "0.60-0.75", "0.75-0.90", "0.90-1.00",
]
FILL_MAX_REL_PRICE_DIFF_BY_BUCKET = {
    "0.00-0.10": 2.50,
    "0.10-0.25": 1.20,
    "0.25-0.40": 0.70,
    "0.40-0.60": 0.35,
    "0.60-0.75": 0.22,
    "0.75-0.90": 0.12,
    "0.90-1.00": 0.06,
}

TRADES_DIR    = Path("../../data/polygon_trades_processed")
WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2")
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

dataset = ds.dataset(TRADES_DIR, format="parquet")
DATASET_COLUMNS        = set(dataset.schema.names)
TRIGGER_TX_HASH_COLUMN = (
    "transaction_hash" if "transaction_hash" in DATASET_COLUMNS
    else ("tx_hash" if "tx_hash" in DATASET_COLUMNS else None)
)

EXEC_TAPE_TEST_PATH    = WORKSPACE_DIR / "execution_tape_test_v2.parquet"
EXEC_TAPE_TRAIN_B_PATH = WORKSPACE_DIR / "execution_tape_train_b_v2.parquet"

BACKTEST_KWARGS = dict(
    base_stake_usdc=100.0,
    latency_seconds=0,
    fill_horizon_seconds=600,
    slippage_bps=50.0,
    fee_bps=10.0,
    max_signals_per_day=20,
    dedupe_by_market=True,
    starting_capital=10_000.0,
    max_rel_price_diff_by_bucket=FILL_MAX_REL_PRICE_DIFF_BY_BUCKET,
)


# Tighter fill parameters for the copy_triggers variant (all event types).
# These enforce strict execution so that marginal add/reduce trades are only
# counted when genuine liquidity is available at the trigger price.
COPY_TRIGGERS_BACKTEST_KWARGS = dict(
    base_stake_usdc=100.0,
    latency_seconds=0,
    fill_horizon_seconds=600,
    slippage_bps=-10.0,
    fee_bps=10.0,
    max_signals_per_day=20,
    dedupe_by_market=True,
    starting_capital=10_000.0,
    max_rel_price_diff_by_bucket={
        "0.00-0.10": 1.00,
        "0.10-0.25": 0.50,
        "0.25-0.40": 0.25,
        "0.40-0.60": 0.12,
        "0.60-0.75": 0.08,
        "0.75-0.90": 0.05,
        "0.90-1.00": 0.03,
    },
)

## Derive train/test dates

In [3]:
_sample = pd.read_parquet(sorted(TRADES_DIR.glob("*.parquet"))[0], columns=["dt", "is_train"])
_sample["dt"] = pd.to_datetime(_sample["dt"], utc=True)
END_DATE_TRAIN   = _sample.loc[_sample["is_train"], "dt"].max().date()
TRAIN_START_DATE = _sample.loc[_sample["is_train"], "dt"].min().date()
TRAIN_A_END_DATE = TRAIN_START_DATE + (END_DATE_TRAIN - TRAIN_START_DATE) // 2
del _sample
print(f"END_DATE_TRAIN={END_DATE_TRAIN}  TRAIN_A_END_DATE={TRAIN_A_END_DATE}")


END_DATE_TRAIN=2026-04-15  TRAIN_A_END_DATE=2025-08-23


## Load strategy registry

In [4]:
from polymarket_analysis.strategy.registry import load_all_strategies
from polymarket_analysis.strategy.triggers import get_trigger

registry_all = load_all_strategies(WORKSPACE_DIR)
registry = {
    sid: s
    for sid, s in registry_all.items()
    if s.selection_method == 'predicting_group' or s.metadata.get('wallet_group') == 'predicting'
}

if not registry:
    raise RuntimeError(
        "No predicting_group strategies found in workspace. "
        "Run stage1_wallet_strategy_selection to save predicting_group strategies first."
    )

summary_rows = []
for sid, s in registry.items():
    summary_rows.append({
        'strategy_id': s.strategy_id,
        'name': s.name,
        'selection_method': s.selection_method,
        'num_wallets': len(s.wallets),
        'trigger_fn': s.trigger.fn_ref.split('.')[-1],
        'threshold': s.trigger.params.get('threshold'),
        'dynamic_sizing': s.trigger.params.get('dynamic_sizing'),
    })

print(f"Loaded {len(registry)} predicting-group strategies (from {len(registry_all)} total):")
pd.DataFrame(summary_rows)


Loaded 3 predicting-group strategies (from 54 total):


,strategy_id,name,selection_method,num_wallets,trigger_fn,threshold,dynamic_sizing
0,predicting_group__all_open_buys,predicting_group | all open-buys (fixed stake),predicting_group,8,all_open_buys,NaN,False
1,predicting_group__copy_triggers,predicting_group | copy all triggers (tight sl...,predicting_group,8,copy_triggers,NaN,False
2,predicting_group__score_threshold,predicting_group | score >= 0.65 (Kelly),predicting_group,8,score_threshold,0.65,True


## Load calibration signals

In [5]:
CALIBRATION_SIGNALS_PATH = WORKSPACE_DIR / "signal_events_train_b.parquet"
TEST_SIGNALS_PATH         = WORKSPACE_DIR / "signal_events_test.parquet"

train_b_signals = pd.read_parquet(CALIBRATION_SIGNALS_PATH)
test_signals    = pd.read_parquet(TEST_SIGNALS_PATH)
print(f"train_b signals: {len(train_b_signals):,}  test signals: {len(test_signals):,}")
print(f"train_b event types: {train_b_signals['event_type'].value_counts().to_dict()}")

train_b signals: 4,789  test signals: 3,943
train_b event types: {'add_buy': 2032, 'reduce_sell': 1939, 'open_buy': 664, 'close_sell': 148, 'other': 6}


## Rebuild calibration tables

In [6]:
from polymarket_analysis.signal.scorer import (
    build_calibration_tables,
    apply_signal_score,
    select_signal_threshold,
)

# Scorer is calibrated on open_buy events only.
train_b_open_buys = train_b_signals[train_b_signals["event_type"] == "open_buy"].copy()

price_table, consensus_table, global_edge = build_calibration_tables(train_b_open_buys)
train_b_scored = apply_signal_score(train_b_open_buys, price_table, consensus_table)
SIGNAL_THRESHOLD = select_signal_threshold(train_b_scored)
print(f"Global edge: {global_edge:.4f}  Signal threshold: {SIGNAL_THRESHOLD:.2f}")

Global edge: 0.2205  Signal threshold: 0.65


## Build per-strategy scored signals

For each strategy, build wallet profiles and signal events **from that strategy's wallet set**,
then score them. This ensures each strategy is evaluated on its own wallet cohort.

In [7]:
from polymarket_analysis.signal.builder import build_wallet_profiles, build_signal_events

strategy_test_signals:      dict[str, pd.DataFrame] = {}
strategy_train_ref_signals: dict[str, pd.DataFrame] = {}

# Group strategies by wallet set identity to avoid redundant profile/signal builds.
# Two strategies share signals when their wallet DataFrames are identical.
_signals_cache: dict[frozenset, tuple[pd.DataFrame, pd.DataFrame]] = {}

for sid, strategy in registry.items():
    wallet_key = frozenset(strategy.wallets["wallet"])
    if wallet_key not in _signals_cache:
        profiles = build_wallet_profiles(
            dataset, strategy.wallets, period="full_train",
            end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE
        )
        _, c_test = build_signal_events(
            dataset, profiles, period="test",
            end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE,
            price_bucket_bins=PRICE_BUCKET_BINS, price_bucket_labels=PRICE_BUCKET_LABELS,
            tx_hash_column=TRIGGER_TX_HASH_COLUMN,
        )
        _, c_train = build_signal_events(
            dataset, profiles, period="train_b",
            end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE,
            price_bucket_bins=PRICE_BUCKET_BINS, price_bucket_labels=PRICE_BUCKET_LABELS,
            tx_hash_column=TRIGGER_TX_HASH_COLUMN,
        )
        scored_test  = apply_signal_score(c_test,  price_table, consensus_table)
        scored_train = apply_signal_score(c_train, price_table, consensus_table)
        _signals_cache[wallet_key] = (scored_test, scored_train)
    strategy_test_signals[sid], strategy_train_ref_signals[sid] = _signals_cache[wallet_key]

{sid: len(df) for sid, df in strategy_test_signals.items()}


{'predicting_group__all_open_buys': 1873,
 'predicting_group__copy_triggers': 1873,
 'predicting_group__score_threshold': 1873}

## Build / load execution tapes

In [8]:
from polymarket_analysis.backtest.execution_tape import (
    build_execution_tape,
    build_tape_groups,
    normalize_execution_tape,
)

def _collect_cids(signals_dict):
    parts = [df["condition_id"] for df in signals_dict.values() if not df.empty]
    if not parts:
        return []
    return list(pd.concat(parts, ignore_index=True).drop_duplicates())

all_test_cids  = _collect_cids(strategy_test_signals)
all_train_cids = _collect_cids(strategy_train_ref_signals)

if EXEC_TAPE_TEST_PATH.exists():
    execution_tape_test = pd.read_parquet(EXEC_TAPE_TEST_PATH)
else:
    execution_tape_test = build_execution_tape(
        dataset, all_test_cids, tx_hash_column=TRIGGER_TX_HASH_COLUMN
    )
    execution_tape_test.to_parquet(EXEC_TAPE_TEST_PATH, index=False)

if EXEC_TAPE_TRAIN_B_PATH.exists():
    execution_tape_train_b = pd.read_parquet(EXEC_TAPE_TRAIN_B_PATH)
else:
    execution_tape_train_b = build_execution_tape(
        dataset, all_train_cids,
        tx_hash_column=TRIGGER_TX_HASH_COLUMN,
        start_date=TRAIN_A_END_DATE + datetime.timedelta(days=1),
        end_date=END_DATE_TRAIN,
    )
    execution_tape_train_b.to_parquet(EXEC_TAPE_TRAIN_B_PATH, index=False)

execution_tape_test["tape_dt"]    = pd.to_datetime(execution_tape_test["tape_dt"],    utc=True)
execution_tape_train_b["tape_dt"] = pd.to_datetime(execution_tape_train_b["tape_dt"], utc=True)
tape_groups           = build_tape_groups(normalize_execution_tape(execution_tape_test))
tape_groups_train_ref = build_tape_groups(normalize_execution_tape(execution_tape_train_b))
print(f"tape markets – test: {len(tape_groups):,}  train_b: {len(tape_groups_train_ref):,}")


tape markets – test: 76,224  train_b: 134,846


## Full strategy sweep

For each strategy:
1. Resolve the trigger function from its `fn_ref`.
2. Apply it to the scored signals to get a boolean mask.
3. Run `backtest_strategy` with `dynamic_sizing` from the strategy's params.


In [9]:
from polymarket_analysis.backtest.strategy import (
    backtest_strategy,
    summarize_backtest,
    build_trigger_ledger,
)

def _run_strategy_registry(registry, signals_dict, tapes):
    """Run backtest for every strategy in the registry.

    Parameters
    ----------
    registry    : {strategy_id -> WalletSelectionStrategy}
    signals_dict: {strategy_id -> scored_signals DataFrame}
    tapes       : tape_groups dict

    Returns
    -------
    runs    : {strategy_id -> {trades, daily, unfilled_triggers, theoretical_daily, spec}}
    summary : DataFrame
    """
    runs, rows = {}, []
    for sid, strategy in registry.items():
        scored_df = signals_dict.get(sid, pd.DataFrame())
        if scored_df.empty:
            continue

        # Resolve and apply trigger function
        trigger_fn = get_trigger(strategy.trigger.fn_ref)
        mask = trigger_fn(scored_df, strategy.trigger.params)

        dynamic_sizing = bool(strategy.trigger.params.get("dynamic_sizing", True))

        # copy_triggers strategies use tighter slippage / fill params
        is_copy_triggers = strategy.trigger.fn_ref.endswith("copy_triggers")
        bt_kwargs = COPY_TRIGGERS_BACKTEST_KWARGS if is_copy_triggers else BACKTEST_KWARGS

        trades_bt, daily_bt, unfilled_bt, theoretical_bt = backtest_strategy(
            scored_df, mask, tapes,
            strategy.strategy_id,
            strategy.trigger.fn_ref.split(".")[-1],
            dynamic_sizing=dynamic_sizing,
            cohort_name=strategy.metadata.get("cohort", strategy.selection_method),
            **bt_kwargs,
        )
        runs[sid] = {
            "trades": trades_bt,
            "daily": daily_bt,
            "unfilled_triggers": unfilled_bt,
            "theoretical_daily": theoretical_bt,
            "strategy": strategy,
        }
        row = summarize_backtest(trades_bt, daily_bt).to_dict()
        n_triggered = len(trades_bt) + len(unfilled_bt)
        row.update({
            "strategy_id": sid,
            "name": strategy.name,
            "selection_method": strategy.selection_method,
            "trigger_fn": strategy.trigger.fn_ref.split(".")[-1],
            "dynamic_sizing": dynamic_sizing,
            "triggered_signals": n_triggered,
            "unfilled_triggers": len(unfilled_bt),
            "fill_rate": len(trades_bt) / n_triggered if n_triggered else float("nan"),
        })
        rows.append(row)

    if not rows:
        return runs, pd.DataFrame()
    summary = (
        pd.DataFrame(rows)
        .set_index("strategy_id")
        .sort_values("net_roi_on_stake", ascending=False)
    )
    return runs, summary


strategy_runs,           strategy_summary           = _run_strategy_registry(
    registry, strategy_test_signals,      tape_groups
)
strategy_runs_train_ref, strategy_summary_train_ref = _run_strategy_registry(
    registry, strategy_train_ref_signals, tape_groups_train_ref
)

trigger_ledgers = {
    n: build_trigger_ledger(r["trades"], r["unfilled_triggers"])
    for n, r in strategy_runs.items()
}

## Summary table

In [10]:
pd.set_option("display.max_columns", None)
strategy_summary


,filled_trades,net_pnl_usdc,net_roi_on_stake,win_rate,name,selection_method,trigger_fn,dynamic_sizing,triggered_signals,unfilled_triggers,fill_rate
strategy_id,,,,,,,,,,,
predicting_group__all_open_buys,0.0,0.0,NaN,NaN,predicting_group | all open-buys (fixed stake),predicting_group,all_open_buys,False,420,420,0.0
predicting_group__copy_triggers,0.0,0.0,NaN,NaN,predicting_group | copy all triggers (tight sl...,predicting_group,copy_triggers,False,420,420,0.0
predicting_group__score_threshold,0.0,0.0,NaN,NaN,predicting_group | score >= 0.65 (Kelly),predicting_group,score_threshold,True,305,305,0.0


## Train-ref summary (in-sample calibration check)

## Cumulative PnL comparison

In [11]:
import pyarrow.dataset as ds
from polymarket_analysis.visualization.backtest_plots import (
    plot_strategy_test,
    plot_strategy_train,
)

# Load stage-0 fills (only the columns we need to keep memory low)
_fills_ds = ds.dataset(TRADES_DIR, format="parquet")
df_fills = _fills_ds.to_table(columns=["wallet", "dt", "trade_pnl"]).to_pandas()
df_fills["dt"] = pd.to_datetime(df_fills["dt"], utc=True)

# Period date bounds (UTC timestamps)
_test_start  = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)
_test_end    = df_fills["dt"].max()
_train_start = pd.Timestamp(TRAIN_START_DATE, tz="UTC")
_train_end   = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(hours=23, minutes=59, seconds=59)

fig_test = plot_strategy_test(
    strategy_runs,
    title="Strategy registry — test period (1h resolution)",
    df_fills=df_fills,
    dt_min=_test_start,
    dt_max=_test_end,
)
fig_test.show(renderer="browser")

fig_train = plot_strategy_train(
    strategy_runs_train_ref,
    title="Strategy registry — train-B reference period (1h resolution)",
    df_fills=df_fills,
    dt_min=_train_start,
    dt_max=_train_end,
)
fig_train.show(renderer="browser")


## Placebo benchmark

Random-wallet baseline to verify the selected strategies beat random chance.

In [12]:
METRICS_FULL_PATH     = WORKSPACE_DIR / 'wallet_metrics_full_train.parquet'
SELECTED_WALLETS_PATH = WORKSPACE_DIR / 'selected_wallets_v2.parquet'

full_train_metrics = pd.read_parquet(METRICS_FULL_PATH)
selected_wallets_all = pd.read_parquet(SELECTED_WALLETS_PATH)

if 'wallet_group' not in selected_wallets_all.columns:
    raise ValueError('selected_wallets_v2.parquet must contain wallet_group labels from stage1.')

selected_wallets = (
    selected_wallets_all[selected_wallets_all['wallet_group'] == 'predicting']
    .drop_duplicates(subset=['wallet'])
    .reset_index(drop=True)
)
if selected_wallets.empty:
    raise ValueError('No predicting wallets found in selected_wallets_v2.parquet.')

rng = np.random.default_rng(42)
eligible_placebo = full_train_metrics[
    (full_train_metrics['open_buy_trades'] >= 20)
    & (full_train_metrics['volume'] >= 500.0)
    & (full_train_metrics.get('distinct_markets', pd.Series(0, index=full_train_metrics.index)) >= 8)
    & (full_train_metrics.get('recent_open_buy_trades', pd.Series(0, index=full_train_metrics.index)) >= 3)
].copy()

SCORE_STRATEGY_ID = 'predicting_group__score_threshold' if 'predicting_group__score_threshold' in strategy_runs else None

if SCORE_STRATEGY_ID and len(eligible_placebo) >= len(selected_wallets):
    placebo_wallets = eligible_placebo.sample(
        n=len(selected_wallets), random_state=42
    ).copy().reset_index(drop=True)
    placebo_wallets['wallet_quality'] = rng.random(len(placebo_wallets))
    placebo_profiles = build_wallet_profiles(
        dataset, placebo_wallets, period='full_train',
        end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE
    )
    _, placebo_test_open_buys = build_signal_events(
        dataset, placebo_profiles, period='test',
        end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE,
        price_bucket_bins=PRICE_BUCKET_BINS, price_bucket_labels=PRICE_BUCKET_LABELS,
        tx_hash_column=TRIGGER_TX_HASH_COLUMN,
    )
    placebo_scored = apply_signal_score(placebo_test_open_buys, price_table, consensus_table)
    if not placebo_scored.empty and 'signal_score' in placebo_scored.columns:
        placebo_trades_bt, placebo_daily_bt, _, _ = backtest_strategy(
            placebo_scored,
            mask=placebo_scored['signal_score'] >= SIGNAL_THRESHOLD,
            tape_groups=tape_groups,
            strategy_name='placebo_random_wallets',
            trigger_rule=f'placebo signal_score >= {SIGNAL_THRESHOLD:.2f}',
            dynamic_sizing=True,
            cohort_name='placebo',
            **BACKTEST_KWARGS,
        )
        test_trades_bt = strategy_runs[SCORE_STRATEGY_ID]['trades']
        test_daily_bt  = strategy_runs[SCORE_STRATEGY_ID]['daily']
        placebo_compare = pd.DataFrame([
            summarize_backtest(test_trades_bt, test_daily_bt).rename(SCORE_STRATEGY_ID),
            summarize_backtest(placebo_trades_bt, placebo_daily_bt).rename('placebo_random_wallets'),
        ])
        placebo_compare
    else:
        print('Placebo signals empty after scoring')
else:
    print('Not enough eligible wallets for placebo or predicting score-threshold strategy not found')


Placebo signals empty after scoring
